# 01 — EDA, Data Quality e Model Readiness

Objetivo: validar estrutura, qualidade, anomalias e prontidão dos dados antes do treinamento.

> Outlier estatístico não é removido automaticamente: pode representar promoção, feriado ou evento real.


In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

from gb_ml.dq import quality_gate, run_quality_checks
from gb_ml.io import sample_table, table_metadata
from gb_ml.profiling import outlier_report, profile_dataframe, temporal_summary

pd.set_option("display.max_columns", 100)
PROJECT_ID = None
TABLE_ID = None  # ex.: "meu-projeto.dataset.vendas"
SAMPLE_LIMIT = 100_000
CONFIG_PATH = Path("../configs/tabelas/exemplo_vendas.yaml")


In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

config


## 1. Data discovery

Primeiro inspecionamos metadados e schema. Para tabelas grandes, prefira agregações no BigQuery antes de trazer dados ao Pandas.


In [ ]:
if TABLE_ID:
    metadata, schema = table_metadata(TABLE_ID, project_id=PROJECT_ID)
    display(pd.DataFrame([metadata]))
    display(schema)
else:
    print("TABLE_ID ainda não configurado. Usando dataset sintético de demonstração.")


## 2. Carga controlada

Enquanto o case real não chega, usamos dados sintéticos propositalmente sujos para testar o pipeline.


In [ ]:
if TABLE_ID:
    df = sample_table(TABLE_ID, project_id=PROJECT_ID, limit=SAMPLE_LIMIT)
else:
    rng = np.random.default_rng(42)
    dates = pd.date_range("2025-01-01", periods=60, freq="D")
    df = pd.DataFrame({
        "data": np.repeat(dates, 2),
        "produto_id": ["A", "B"] * len(dates),
        "loja_id": ["L01"] * (len(dates) * 2),
        "vendas": np.maximum(0, rng.normal(100, 20, len(dates) * 2)).round(0),
        "preco": rng.normal(25, 2, len(dates) * 2).round(2),
    })
    df.loc[5, "vendas"] = 2500
    df.loc[8, "vendas"] = -10
    df.loc[10, "preco"] = np.nan
    df = pd.concat([df, df.iloc[[0]]], ignore_index=True)

df.head()


## 3. Sanity check estrutural


In [ ]:
print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]:,}")
print(f"Memória aproximada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(df.dtypes.rename("dtype").to_frame())


## 4. Profiling automático por coluna


In [ ]:
profile = profile_dataframe(df)
display(profile)


## 5. Data Quality e Quality Gate

Checks `warn` pedem investigação; checks `error` bloqueiam o treino até serem entendidos ou corrigidos.


In [ ]:
dq_report = run_quality_checks(df, config)
display(dq_report)

approved, critical_failures = quality_gate(dq_report)
print("QUALITY GATE:", "APROVADO" if approved else "BLOQUEADO")
if not approved:
    display(critical_failures)


## 6. Outliers robustos

MAD é usado como detector robusto. O objetivo é sinalizar para investigação, não excluir automaticamente.


In [ ]:
outliers = outlier_report(df, method="mad", threshold=3.5)
display(outliers.sort_values("pct_outliers", ascending=False))


## 7. Cobertura temporal


In [ ]:
date_columns = [
    col for col, rules in config.get("columns", {}).items()
    if rules.get("type") in {"date", "datetime", "timestamp"} and col in df.columns
]

if date_columns:
    date_col = date_columns[0]
    group_cols = [c for c in config.get("grain", []) if c != date_col and c in df.columns]
    temporal = temporal_summary(df, date_col, group_columns=group_cols[:2], freq="D")
    display(temporal.head(20))


## 8. Relações numéricas

Correlação é triagem exploratória, não causalidade. Para forecasting, toda feature também precisa passar por auditoria de disponibilidade temporal para evitar leakage.


In [ ]:
numeric_cols = list(df.select_dtypes(include="number").columns)
if len(numeric_cols) >= 2:
    display(df[numeric_cols].corr(method="spearman"))


## 9. Checklist de Model Readiness

Antes do treino, documentar:

1. Target exato.
2. Grão da previsão.
3. Horizonte de forecast.
4. Cutoff de informação.
5. Riscos de leakage.
6. Split temporal, nunca aleatório para forecast.
7. Baseline obrigatório.
8. Métricas adequadas (MAE, WAPE, bias, skill vs baseline).
9. Quality gate sem falhas críticas.

A saída final do EDA deve separar **problemas críticos**, **warnings** e **insights analíticos**.
